# Max Independent Set: a native quantum optimization workflow

Preserve the original 8-node seeded graph, three layers, 60-iteration bound and quantile 0.7. Native qfunc expresses the retained score and mixer. The explicit native optimizer/readout replaces foreign modeling and execution services.


In [ ]:
import matplotlib.pyplot as plt

import flagquantum as fq
from examples.synthesis.applications.optimization.max_independent_set.overview.workflow import (
    run_workflow,
    source_edges,
)
from examples.synthesis.graph_optimization_plot import plot_workflow
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def cost_layer(gamma: float, v: QArray[QBit]) -> None:
    phase(
        v[0]
        + v[1]
        + v[2]
        + v[3]
        + v[4]
        + v[5]
        + v[6]
        + v[7]
        - (
            v[0] * v[2]
            + v[0] * v[4]
            + v[0] * v[6]
            + v[0] * v[7]
            + v[1] * v[2]
            + v[1] * v[4]
            + v[1] * v[5]
            + v[2] * v[4]
            + v[2] * v[5]
            + v[2] * v[6]
            + v[3] * v[4]
            + v[3] * v[5]
            + v[3] * v[6]
            + v[5] * v[6]
        ),
        gamma,
    )


@qfunc
def main(params: list[float], v: Output[QArray[QBit]]) -> None:
    allocate(8, v)
    hadamard_transform(v)
    for i in range(3):
        cost_layer(params[i], v)
        for j in range(8):
            RX(params[3 + i], v[j])


In [ ]:
edges = source_edges()
print("original graph edges", edges)
program = synth.synthesize(
    main,
    classical_inputs={"params": (0.13, -0.27, 0.19, 0.22, 0.31, -0.17)},
    max_qubits=8, max_workspace=0, max_operations=5000,
    memory_limit_bytes=4 * 1024**2,
)
print(dict(program.probabilities(fq.run(program), registers=("v",))))
print(program.report["resources"])


## Optimize actual native executions

Every objective call synthesizes and runs this quantum program. We minimize lower-tail penalized CVaR at quantile 0.7 with the public Nelder-Mead optimizer. The original iteration limit remains in place; convergence is reported rather than assumed. No exhaustive answer enters the optimizer or sample selection.


In [ ]:
workflow = run_workflow()
print("initial CVaR", workflow.trace[0].cost)
print("final CVaR", workflow.optimizer.value)
print("iterations", workflow.optimizer.iterations)
print("actual quantum evaluations", workflow.optimizer.evaluations)
print("converged", workflow.optimizer.converged)
print("best observed feasible", workflow.best_feasible)
print("exact bounded classical reference", workflow.exact_nodes)
for row in workflow.optimized[:5]:
    print(row)


## Compare sampling and the labeled classical reference

Draw 1000 optimized local samples with seed 2718 and 1000 uniform samples with seed 31415. Filter feasibility before selecting an answer: a low penalized cost alone need not imply a valid subset. Histograms show the unpenalized negative subset size, including infeasible observations. The bounded exhaustive reference replaces the external classical solver only for this small fixed problem. A seeded force layout replaces Kamada-Kawai; it preserves graph data.


In [ ]:
figure = plot_workflow(workflow, edges, seed=12345, title="Max Independent Set: native workflow")
plt.show()
plt.close(figure)
